# LeetCode 747 — Largest Number At Least Twice of Others bằng Python

**Link đề:** https://leetcode.com/problems/largest-number-at-least-twice-of-others/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting

```python
class Solution:
    def dominantIndex(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đọc đúng yêu cầu: trả **chỉ số** của phần tử lớn nhất, và chỉ khi nó **≥ 2 lần mọi phần tử khác**.
2. Ôn `max`, `list.index`, `enumerate`, `sorted`, `heapq.nlargest`.
3. Biến điều kiện "≥ 2 lần **mọi** phần tử khác" thành "≥ 2 lần **phần tử lớn thứ hai**" — rút gọn bài toán.
4. Viết 4 cách: hai lượt (theo hint) → sắp xếp → `heapq` → **một lượt giữ top-2** (lời giải chính).
5. Trace, chứng minh, test đối chiếu, luyện biến thể (top-k, lớn thứ ba, k lần).

## 1. Đề bài

Cho mảng số nguyên `nums`, trong đó **phần tử lớn nhất là duy nhất**.

Kiểm tra phần tử lớn nhất có **lớn hơn hoặc bằng 2 lần** mọi phần tử khác không. Nếu có → trả về **chỉ số** của phần tử lớn nhất; nếu không → trả `-1`.

| nums | Output | Vì sao |
|---|:---:|---|
| `[3,6,1,0]` | `1` | max = 6 (ở chỉ số 1); `6 >= 2·3`, `6 >= 2·1`, `6 >= 2·0` |
| `[1,2,3,4]` | `-1` | max = 4 nhưng `4 < 2·3 = 6` |

**Constraints:** `2 <= len(nums) <= 50`, `0 <= nums[i] <= 100`, phần tử lớn nhất là **duy nhất**.

Chú ý: điều kiện là `>=` ("at least twice") — `max == 2·x` vẫn hợp lệ. Ví dụ `[1, 2]` → `2 >= 2·1` ✅ → trả `1`.

In [1]:
nums = [3, 6, 1, 0]
m = max(nums)
print("max =", m)
for i, x in enumerate(nums):
    if x != m:
        print(f"  {m} >= 2*{x} = {2 * x}? {m >= 2 * x}")

max = 6
  6 >= 2*3 = 6? True
  6 >= 2*1 = 2? True
  6 >= 2*0 = 0? True


## 2. Nền tảng Python

### 2.1. `max()` và `list.index()`

- `max(nums)` trả **giá trị** lớn nhất (`O(n)`).
- `nums.index(v)` trả **chỉ số đầu tiên** có giá trị `v` (`O(n)`); ném `ValueError` nếu không có.

In [2]:
nums = [3, 6, 1, 0]
print(max(nums), nums.index(max(nums)))
try:
    nums.index(99)
except ValueError as e:
    print("ValueError:", e)

6 1
ValueError: 99 is not in list


### 2.2. `max` với `key` để lấy chỉ số trực tiếp

`max(range(len(nums)), key=lambda i: nums[i])` duyệt các **chỉ số**, nhưng so sánh theo `nums[i]` → trả về **chỉ số** của phần tử lớn nhất trong một lượt.

In [3]:
nums = [3, 6, 1, 0]
best_i = max(range(len(nums)), key=lambda i: nums[i])
print("chỉ số max:", best_i, "| giá trị:", nums[best_i])

chỉ số max: 1 | giá trị: 6


### 2.3. `sorted()` khác `list.sort()`

- `sorted(nums)` trả **list mới** đã sắp xếp, `nums` giữ nguyên → không mất thông tin chỉ số gốc.
- `nums.sort()` sắp xếp **tại chỗ** và trả `None`.
- `reverse=True` để sắp giảm dần.

In [4]:
nums = [3, 6, 1, 0]
s = sorted(nums, reverse=True)
print("sorted:", s, "| nums vẫn là:", nums)
r = nums.sort()
print("sort() trả:", r, "| nums giờ là:", nums)

sorted: [6, 3, 1, 0] | nums vẫn là: [3, 6, 1, 0]
sort() trả: None | nums giờ là: [0, 1, 3, 6]


### 2.4. `heapq.nlargest(k, iterable)`

Lấy `k` phần tử lớn nhất (giảm dần) mà không cần sắp xếp cả mảng — `O(n log k)`.

In [5]:
import heapq
print(heapq.nlargest(2, [3, 6, 1, 0]))
print(heapq.nlargest(3, [1, 2, 3, 4]))

[6, 3]
[4, 3, 2]


## 3. Cách 1 — Hai lượt (đúng theo hint của đề)

1. Lượt 1: tìm `m = max(nums)` và chỉ số `max_i`.
2. Lượt 2: với mọi `x` **khác `m`**, nếu `m < 2·x` → trả `-1`.
3. Qua hết → trả `max_i`.

In [6]:
def dominant_two_pass(nums: list[int]) -> int:
    m = max(nums)
    max_i = nums.index(m)
    for i, x in enumerate(nums):
        if i != max_i and m < 2 * x:
            return -1
    return max_i

print(dominant_two_pass([3, 6, 1, 0]))   # 1
print(dominant_two_pass([1, 2, 3, 4]))   # -1
print(dominant_two_pass([1, 2]))         # 1

1
-1
1


So sánh `i != max_i` (theo **chỉ số**) an toàn hơn `x != m` (theo **giá trị**): nếu bỏ giả thiết "max duy nhất", so theo giá trị sẽ bỏ qua cả phần tử trùng max — điều có thể không mong muốn. Với đề này cả hai đều đúng.

Cách 1 đã `O(n)` thời gian, `O(1)` bộ nhớ. Câu hỏi để học sâu hơn: **ta có cần so với mọi phần tử không?**

## 4. Insight — chỉ cần phần tử lớn thứ hai

"`m >= 2·x` với **mọi** `x` khác" ⟺ "`m >= 2·x` với `x` **lớn nhất** trong số đó" — vì `2·x` lớn nhất khi `x` lớn nhất. Phần tử lớn nhất trong các phần tử còn lại chính là **phần tử lớn thứ hai** `second`.

```text
m >= 2 * second   =>   m >= 2 * second >= 2 * x   với mọi x khác
```

Ví dụ `[3, 6, 1, 0]`: `second = 3`, `6 >= 6` ✅. Ví dụ `[1, 2, 3, 4]`: `second = 3`, `4 >= 6` ❌.

Vậy bài toán trở thành: **tìm hai phần tử lớn nhất**.

In [7]:
for nums in [[3, 6, 1, 0], [1, 2, 3, 4], [1, 2], [0, 0, 3, 2]]:
    top, second = heapq.nlargest(2, nums)
    print(nums, "-> top =", top, ", second =", second, ", top >= 2*second?", top >= 2 * second)

[3, 6, 1, 0] -> top = 6 , second = 3 , top >= 2*second? True
[1, 2, 3, 4] -> top = 4 , second = 3 , top >= 2*second? False
[1, 2] -> top = 2 , second = 1 , top >= 2*second? True
[0, 0, 3, 2] -> top = 3 , second = 2 , top >= 2*second? False


## 5. Cách 2 — Sắp xếp

Sắp xếp **bản sao** giảm dần, lấy `s[0]` (max) và `s[1]` (lớn thứ hai). Cần chỉ số gốc → `nums.index(s[0])`.

In [8]:
def dominant_sort(nums: list[int]) -> int:
    s = sorted(nums, reverse=True)
    if s[0] >= 2 * s[1]:
        return nums.index(s[0])
    return -1

print(dominant_sort([3, 6, 1, 0]), dominant_sort([1, 2, 3, 4]), dominant_sort([1, 2]))

1 -1 1


Gọn nhưng tốn `O(n log n)` cho việc sắp xếp — làm thừa, vì ta chỉ cần 2 phần tử đầu.

## 6. Cách 3 — `heapq.nlargest(2, ...)`

Lấy đúng 2 phần tử lớn nhất trong `O(n log 2) = O(n)`.

In [9]:
def dominant_heap(nums: list[int]) -> int:
    top, second = heapq.nlargest(2, nums)
    return nums.index(top) if top >= 2 * second else -1

print(dominant_heap([3, 6, 1, 0]), dominant_heap([1, 2, 3, 4]), dominant_heap([1, 2]))

1 -1 1


`a if cond else b` là **biểu thức điều kiện** (ternary): trả `a` khi `cond` đúng, ngược lại `b`.

## 7. Cách 4 — Một lượt, giữ `first` và `second` (lời giải chính)

Duyệt một lần, duy trì:
- `first` = giá trị lớn nhất đã thấy, `first_i` = chỉ số của nó;
- `second` = giá trị lớn thứ hai đã thấy.

Khi gặp `x` tại `i`:
- Nếu `x > first`: phần tử lớn nhất cũ **bị đẩy xuống** hạng hai → `second = first`; rồi `first, first_i = x, i`.
- Ngược lại nếu `x > second`: chỉ cập nhật `second = x`.
- Ngược lại: không làm gì.

Khởi tạo `first = second = -1` (nhỏ hơn mọi giá trị hợp lệ vì `nums[i] >= 0`).

In [10]:
def dominant_one_pass(nums: list[int]) -> int:
    first, second, first_i = -1, -1, -1
    for i, x in enumerate(nums):
        if x > first:
            second = first
            first, first_i = x, i
        elif x > second:
            second = x
    return first_i if first >= 2 * second else -1

print(dominant_one_pass([3, 6, 1, 0]), dominant_one_pass([1, 2, 3, 4]), dominant_one_pass([1, 2]))

1 -1 1


Thứ tự hai lệnh trong nhánh `x > first` **rất quan trọng**: phải gán `second = first` **trước** khi ghi đè `first`, nếu không giá trị max cũ bị mất.

`elif` = "else if": chỉ xét khi nhánh `if` phía trên sai.

## 8. Trace từng bước

`nums = [3, 6, 1, 0]`:

| i | x | So sánh | first (sau) | first_i | second (sau) |
|---:|---:|---|---:|---:|---:|
| — | — | khởi tạo | -1 | -1 | -1 |
| 0 | 3 | 3 > -1 → max mới | 3 | 0 | -1 |
| 1 | 6 | 6 > 3 → max mới, 3 xuống hạng 2 | 6 | 1 | 3 |
| 2 | 1 | 1 ≤ 6, 1 ≤ 3 → bỏ | 6 | 1 | 3 |
| 3 | 0 | bỏ | 6 | 1 | 3 |

Cuối: `6 >= 2·3 = 6` ✅ → trả `1`.

In [11]:
def trace_dominant(nums):
    first, second, first_i = -1, -1, -1
    print(f"nums = {nums}")
    for i, x in enumerate(nums):
        if x > first:
            action = f"{x} > first={first} -> max mới, {first} xuống hạng 2"
            second = first
            first, first_i = x, i
        elif x > second:
            action = f"{x} > second={second} -> hạng 2 mới"
            second = x
        else:
            action = "bỏ qua"
        print(f"i={i} x={x:>3} | {action:<40} | first={first:>3} (i={first_i}) second={second:>3}")
    ok = first >= 2 * second
    print(f"Kết luận: {first} >= 2*{second} = {2 * second}? {ok} -> trả {first_i if ok else -1}")
    return first_i if ok else -1

trace_dominant([3, 6, 1, 0])
print()
trace_dominant([1, 2, 3, 4])

nums = [3, 6, 1, 0]
i=0 x=  3 | 3 > first=-1 -> max mới, -1 xuống hạng 2 | first=  3 (i=0) second= -1
i=1 x=  6 | 6 > first=3 -> max mới, 3 xuống hạng 2   | first=  6 (i=1) second=  3
i=2 x=  1 | bỏ qua                                   | first=  6 (i=1) second=  3
i=3 x=  0 | bỏ qua                                   | first=  6 (i=1) second=  3
Kết luận: 6 >= 2*3 = 6? True -> trả 1

nums = [1, 2, 3, 4]
i=0 x=  1 | 1 > first=-1 -> max mới, -1 xuống hạng 2 | first=  1 (i=0) second= -1
i=1 x=  2 | 2 > first=1 -> max mới, 1 xuống hạng 2   | first=  2 (i=1) second=  1
i=2 x=  3 | 3 > first=2 -> max mới, 2 xuống hạng 2   | first=  3 (i=2) second=  2
i=3 x=  4 | 4 > first=3 -> max mới, 3 xuống hạng 2   | first=  4 (i=3) second=  3
Kết luận: 4 >= 2*3 = 6? False -> trả -1


-1

### Trace khi max nằm ở đầu — nhánh `elif` làm việc

Với `[10, 2, 5, 4]`: `first` không đổi sau bước 0, nên các phần tử sau chỉ tranh hạng hai. Nếu quên nhánh `elif`, `second` sẽ kẹt ở `-1` và ta trả lời sai.

In [12]:
trace_dominant([10, 2, 5, 4])
print()
trace_dominant([10, 2, 6, 4])

nums = [10, 2, 5, 4]
i=0 x= 10 | 10 > first=-1 -> max mới, -1 xuống hạng 2 | first= 10 (i=0) second= -1
i=1 x=  2 | 2 > second=-1 -> hạng 2 mới              | first= 10 (i=0) second=  2
i=2 x=  5 | 5 > second=2 -> hạng 2 mới               | first= 10 (i=0) second=  5
i=3 x=  4 | bỏ qua                                   | first= 10 (i=0) second=  5
Kết luận: 10 >= 2*5 = 10? True -> trả 0

nums = [10, 2, 6, 4]
i=0 x= 10 | 10 > first=-1 -> max mới, -1 xuống hạng 2 | first= 10 (i=0) second= -1
i=1 x=  2 | 2 > second=-1 -> hạng 2 mới              | first= 10 (i=0) second=  2
i=2 x=  6 | 6 > second=2 -> hạng 2 mới               | first= 10 (i=0) second=  6
i=3 x=  4 | bỏ qua                                   | first= 10 (i=0) second=  6
Kết luận: 10 >= 2*6 = 12? False -> trả -1


-1

## 9. Code nộp LeetCode

### Một lượt (khuyến nghị)

In [13]:
class Solution:
    def dominantIndex(self, nums: list[int]) -> int:
        first = second = -1
        first_i = -1
        for i, x in enumerate(nums):
            if x > first:
                second = first
                first, first_i = x, i
            elif x > second:
                second = x
        return first_i if first >= 2 * second else -1


sol = Solution()
print(sol.dominantIndex([3, 6, 1, 0]), sol.dominantIndex([1, 2, 3, 4]))

1 -1


`first = second = -1` là **gán chuỗi**: cả hai tên cùng trỏ tới số `-1`. An toàn với số (bất biến); **đừng** dùng kiểu này với list (`a = b = []` → hai tên cùng một list).

### Các bản khác (đổi tên class thành `Solution` khi nộp)

In [14]:
class SolutionTwoPass:
    def dominantIndex(self, nums: list[int]) -> int:
        m = max(nums)
        idx = nums.index(m)
        return idx if all(m >= 2 * x for i, x in enumerate(nums) if i != idx) else -1


class SolutionSort:
    def dominantIndex(self, nums: list[int]) -> int:
        s = sorted(nums)
        return nums.index(s[-1]) if s[-1] >= 2 * s[-2] else -1


print(SolutionTwoPass().dominantIndex([3, 6, 1, 0]), SolutionSort().dominantIndex([1, 2, 3, 4]))

1 -1


`s[-1]`, `s[-2]` là phần tử cuối và kế cuối của list tăng dần → max và lớn thứ hai.

## 10. Vì sao đúng?

**Rút gọn điều kiện.** Gọi `m` là max (duy nhất), `s` là max của các phần tử còn lại. Với mọi `x` khác: `x <= s` ⇒ `2x <= 2s`. Do đó `m >= 2s` ⟹ `m >= 2x` với mọi `x`; ngược lại nếu `m < 2s` thì chính `x = s` vi phạm. Vậy điều kiện của đề ⟺ `m >= 2s`.

**Invariant của Cách 4.** Sau khi xử lý `nums[0..i]`: `first` = max của đoạn đó (`first_i` là chỉ số của nó), `second` = max của đoạn đó **bỏ đi** phần tử ở `first_i` (hoặc `-1` nếu đoạn chỉ có 1 phần tử).
- `x > first`: `x` thành max mới; max cũ `first` lớn hơn hoặc bằng mọi phần tử khác cũ → trở thành hạng hai đúng.
- `second < x <= first`: `x` không phải max nhưng lớn hơn hạng hai cũ → hạng hai mới.
- `x <= second`: không ảnh hưởng hai hạng đầu.

Kết thúc vòng lặp: `first = m`, `second = s` → so `first >= 2 * second` là đúng điều kiện.

**Vì sao `-1` làm giá trị khởi tạo được?** Mọi `nums[i] >= 0 > -1`, nên phần tử đầu luôn vào nhánh `x > first`. Và vì `n >= 2`, `second` chắc chắn được cập nhật thành một phần tử thật.

## 11. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Hai lượt | `O(n)` (2 lượt + `index`) | `O(1)` |
| 2. Sắp xếp | `O(n log n)` | `O(n)` bản sao |
| 3. `heapq.nlargest(2)` | `O(n)` | `O(1)` |
| 4. Một lượt top-2 | `O(n)` (đúng 1 lượt) | `O(1)` |

Với `n <= 50`, mọi cách đều tức thì. Giá trị của bài nằm ở **kỹ thuật giữ top-2 trong một lượt** — dùng lại cho rất nhiều bài (Third Maximum Number, Maximum Product of Two Elements, ...).

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("đề 1", sol.dominantIndex([3, 6, 1, 0]), 1)
check("đề 2", sol.dominantIndex([1, 2, 3, 4]), -1)
check("vừa đúng 2 lần", sol.dominantIndex([1, 2]), 1)
check("thiếu 1 chút", sol.dominantIndex([2, 3]), -1)
check("có số 0", sol.dominantIndex([0, 1]), 1)
check("max ở đầu", sol.dominantIndex([10, 2, 5, 4]), 0)
check("max ở đầu, sai", sol.dominantIndex([10, 2, 6, 4]), -1)
check("max ở cuối", sol.dominantIndex([1, 0, 2, 0, 4]), 4)
check("hạng 2 lặp lại", sol.dominantIndex([3, 3, 7]), 2)
check("hạng 2 lặp lại, đạt", sol.dominantIndex([3, 3, 6]), 2)

PASS  đề 1: 1
PASS  đề 2: -1
PASS  vừa đúng 2 lần: 1
PASS  thiếu 1 chút: -1
PASS  có số 0: 1
PASS  max ở đầu: 0
PASS  max ở đầu, sai: -1
PASS  max ở cuối: 4
PASS  hạng 2 lặp lại: 2
PASS  hạng 2 lặp lại, đạt: 2


In [16]:
import random

rng = random.Random(747)
impls = [dominant_two_pass, dominant_sort, dominant_heap, dominant_one_pass,
         Solution().dominantIndex, SolutionTwoPass().dominantIndex, SolutionSort().dominantIndex]
hits = 0
for _ in range(5000):
    n = rng.randint(2, 8)
    nums = [rng.randint(0, 20) for _ in range(n - 1)]
    nums.insert(rng.randint(0, n - 1), max(nums) + rng.randint(1, 25))   # bảo đảm max duy nhất
    outs = {f(nums) for f in impls}
    assert len(outs) == 1, (nums, outs)
    hits += outs.pop() != -1
print("5000 mảng ngẫu nhiên (max duy nhất): 7 cài đặt khớp nhau; số mảng có đáp án khác -1:", hits)

5000 mảng ngẫu nhiên (max duy nhất): 7 cài đặt khớp nhau; số mảng có đáp án khác -1: 2139


Ta **sinh dữ liệu đúng constraint**: chèn một phần tử lớn hơn hẳn mọi phần tử khác để max luôn duy nhất. Test với dữ liệu vi phạm constraint có thể làm các cách "đúng" cho kết quả khác nhau mà không phải lỗi.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Trả giá trị max thay vì chỉ số

In [17]:
def wrong_return_value(nums):
    m = max(nums)
    return m if all(m >= 2 * x for x in nums if x != m) else -1

print("sai :", wrong_return_value([3, 6, 1, 0]), "| đúng:", dominant_one_pass([3, 6, 1, 0]))

sai : 6 | đúng: 1


### Lỗi 2: Quên loại chính max khi so sánh

`m >= 2 * m` chỉ đúng khi `m == 0` → hầu như luôn trả `-1`.

In [18]:
def wrong_include_self(nums):
    m = max(nums)
    return nums.index(m) if all(m >= 2 * x for x in nums) else -1

print("sai :", wrong_include_self([3, 6, 1, 0]), "| đúng:", dominant_one_pass([3, 6, 1, 0]))

sai : -1 | đúng: 1


### Lỗi 3: Dùng `>` thay vì `>=`

In [19]:
def wrong_strict(nums):
    first, second, fi = -1, -1, -1
    for i, x in enumerate(nums):
        if x > first:
            second, first, fi = first, x, i
        elif x > second:
            second = x
    return fi if first > 2 * second else -1       # SAI: phải là >=

print("sai :", wrong_strict([3, 6, 1, 0]), "| đúng:", dominant_one_pass([3, 6, 1, 0]))

sai : -1 | đúng: 1


### Lỗi 4: Thiếu nhánh `elif` — chỉ cập nhật `second` khi có max mới

Khi max xuất hiện **trước** phần tử lớn thứ hai, `second` không bao giờ được cập nhật đúng.

In [20]:
def wrong_no_elif(nums):
    first, second, fi = -1, -1, -1
    for i, x in enumerate(nums):
        if x > first:
            second = first
            first, fi = x, i
    return fi if first >= 2 * second else -1

print("sai :", wrong_no_elif([10, 2, 6, 4]), "| đúng:", dominant_one_pass([10, 2, 6, 4]))

sai : 0 | đúng: -1


### Lỗi 5: Ghi đè `first` trước khi lưu vào `second`

In [21]:
def wrong_order(nums):
    first, second, fi = -1, -1, -1
    for i, x in enumerate(nums):
        if x > first:
            first, fi = x, i
            second = first           # SAI: second nhận luôn giá trị mới
        elif x > second:
            second = x
    return fi if first >= 2 * second else -1

print("sai :", wrong_order([3, 6, 1, 0]), "| đúng:", dominant_one_pass([3, 6, 1, 0]))

sai : -1 | đúng: 1


**Sửa:** `second = first` trước, hoặc gán đồng thời `second, first, first_i = first, x, i`.

## 14. Mẹo nhận dạng pattern

- "So với **mọi** phần tử khác" mà điều kiện **đơn điệu** (càng lớn càng khó thỏa) → chỉ cần so với phần tử **cực trị** (ở đây: lớn thứ hai).
- Cần **top-k với k nhỏ** (1, 2, 3) → giữ `k` biến trong một lượt, cập nhật kiểu "đẩy xuống hạng". `k` lớn hơn → `heapq`.
- Đề hỏi **chỉ số** → lưu chỉ số song song với giá trị.
- Bài cùng họ: 414 Third Maximum Number, 1464 Maximum Product of Two Elements in an Array, 628 Maximum Product of Three Numbers, 215 Kth Largest Element.

## 15. Biến thể

### 15.1. "Gấp `k` lần" thay vì 2 lần

Chỉ thay hằng số. Với `k` thực (ví dụ `1.5`) cũng chạy.

In [22]:
def dominant_k_times(nums, k):
    first, second, fi = float("-inf"), float("-inf"), -1
    for i, x in enumerate(nums):
        if x > first:
            second, first, fi = first, x, i
        elif x > second:
            second = x
    return fi if first >= k * second else -1

print(dominant_k_times([3, 6, 1, 0], 2), dominant_k_times([3, 6, 1, 0], 3), dominant_k_times([1, 2, 3, 4], 1.3))

1 -1 3


Dùng `float("-inf")` (âm vô cùng) để khởi tạo — an toàn cả khi mảng có số âm (bài gốc không có).

### 15.2. LeetCode 1464 — Maximum Product of Two Elements

Trả `(a - 1) * (b - 1)` lớn nhất với `a, b` là hai phần tử ở **hai chỉ số khác nhau**. Vì mọi phần tử ≥ 1, đáp án dùng hai phần tử lớn nhất — đúng khung top-2.

In [23]:
def max_product(nums):
    first = second = 0
    for x in nums:
        if x > first:
            second, first = first, x
        elif x > second:
            second = x
    return (first - 1) * (second - 1)

print(max_product([3, 4, 5, 2]), max_product([1, 5, 4, 5]), max_product([3, 7]))   # 12 16 12

12 16 12


Chú ý `[1, 5, 4, 5]`: hai số `5` ở **hai chỉ số khác nhau** → `second` phải nhận được `5`. Với khung `x > first / elif x > second`, `5` thứ hai không `> first = 5` nhưng `> second = 4` → vào `elif` ✔.

## 16. Bài tập

### Bài 1 — Trace bằng tay
Với `nums = [4, 9, 2, 4, 1]`, lập bảng `first, first_i, second` theo Cách 4. Kết quả?

### Bài 2 — Hai chỉ số lớn nhất
Trả `(i1, i2)`: chỉ số của phần tử lớn nhất và lớn thứ hai (giả sử các giá trị đôi một khác nhau), trong **một lượt**.

In [24]:
def top2_indices(nums: list[int]) -> tuple[int, int]:
    # TODO
    pass

# assert top2_indices([3, 6, 1, 0]) == (1, 0)
# assert top2_indices([1, 2, 3, 4]) == (3, 2)

### Bài 3 — Điền chỗ trống lời giải chính

In [25]:
def exercise_dominant(nums: list[int]) -> int:
    first = second = -1
    first_i = -1
    for i, x in enumerate(nums):
        # TODO 1: nếu x > first: đẩy first xuống second, cập nhật first và first_i
        # TODO 2: elif x > second: cập nhật second
        pass
    # TODO 3: trả first_i nếu first >= 2 * second, ngược lại -1
    pass

# assert exercise_dominant([3, 6, 1, 0]) == 1
# assert exercise_dominant([1, 2, 3, 4]) == -1
# assert exercise_dominant([10, 2, 6, 4]) == -1

### Bài 4 — Third Maximum Number (LeetCode 414)
Trả về giá trị lớn thứ ba **trong các giá trị phân biệt**; nếu không có (ít hơn 3 giá trị phân biệt) thì trả giá trị lớn nhất. Một lượt, `O(1)` bộ nhớ.

In [26]:
def third_max(nums: list[int]) -> int:
    # TODO
    pass

# assert third_max([3, 2, 1]) == 1
# assert third_max([1, 2]) == 2
# assert third_max([2, 2, 3, 1]) == 1
# assert third_max([-5, -1, -3]) == -5

### Bài 5 — Phần tử "thống trị" theo tổng
Trả chỉ số của phần tử lớn nhất nếu nó **lớn hơn hoặc bằng tổng** mọi phần tử còn lại, ngược lại `-1` (max duy nhất).

In [27]:
def dominant_by_sum(nums: list[int]) -> int:
    # TODO
    pass

# assert dominant_by_sum([1, 2, 6]) == 2
# assert dominant_by_sum([1, 2, 2, 4]) == -1
# assert dominant_by_sum([5, 0, 0]) == 0

### Bài 6 — Mini challenge: Maximum Product of Three Numbers (LeetCode 628)
Mảng có thể có số âm. Tìm tích lớn nhất của 3 phần tử. Gợi ý: không chỉ top-3 lớn nhất.

In [28]:
def max_product_three(nums: list[int]) -> int:
    # TODO
    pass

# assert max_product_three([1, 2, 3]) == 6
# assert max_product_three([1, 2, 3, 4]) == 24
# assert max_product_three([-10, -10, 1, 3, 2]) == 300

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Max xuất hiện ở bước 1 | Sau đó chỉ còn tranh hạng 2 | `first=9 (i=1)`, `second=4` → `9 >= 8` |
| 2 | Giữ chỉ số thay vì giá trị | `i1, i2` khởi tạo `-1` | `if i1 == -1 or x > nums[i1]: i2, i1 = i1, i` ... |
| 3 | Như Cách 4 | `second = first` trước | `return first_i if first >= 2 * second else -1` |
| 4 | Ba biến, bỏ qua giá trị trùng | `if x in (a, b, c): continue` | khởi tạo `None` hoặc `-inf`, đẩy xuống từng hạng |
| 5 | Tổng còn lại = tổng - max | `total = sum(nums)` | `m >= total - m` |
| 6 | Hai số âm nhân nhau thành dương | Cần top-3 lớn và top-2 nhỏ | `max(a*b*c, a*min1*min2)` |

## 18. Đáp án tham khảo

**Bài 1:** `[4, 9, 2, 4, 1]`: i=0 → first=4; i=1 → 9 > 4 → second=4, first=9 (i=1); i=2 → bỏ; i=3 → `4 > 4`? không → bỏ; i=4 → bỏ. `9 >= 8` ✅ → trả `1`.

In [29]:
trace_dominant([4, 9, 2, 4, 1])


def answer_top2_indices(nums):
    i1 = i2 = -1
    for i, x in enumerate(nums):
        if i1 == -1 or x > nums[i1]:
            i2, i1 = i1, i
        elif i2 == -1 or x > nums[i2]:
            i2 = i
    return i1, i2


def answer_exercise_dominant(nums):
    first = second = -1
    first_i = -1
    for i, x in enumerate(nums):
        if x > first:
            second = first
            first, first_i = x, i
        elif x > second:
            second = x
    return first_i if first >= 2 * second else -1


def answer_third_max(nums):
    a = b2 = c = None                     # a > b2 > c
    for x in nums:
        if x in (a, b2, c):
            continue
        if a is None or x > a:
            a, b2, c = x, a, b2
        elif b2 is None or x > b2:
            b2, c = x, b2
        elif c is None or x > c:
            c = x
    return a if c is None else c


def answer_dominant_by_sum(nums):
    m = max(nums)
    return nums.index(m) if m >= sum(nums) - m else -1


def answer_max_product_three(nums):
    top3 = heapq.nlargest(3, nums)
    low2 = heapq.nsmallest(2, nums)
    return max(top3[0] * top3[1] * top3[2], top3[0] * low2[0] * low2[1])


assert answer_top2_indices([3, 6, 1, 0]) == (1, 0)
assert answer_top2_indices([1, 2, 3, 4]) == (3, 2)
assert answer_exercise_dominant([3, 6, 1, 0]) == 1
assert answer_exercise_dominant([1, 2, 3, 4]) == -1
assert answer_exercise_dominant([10, 2, 6, 4]) == -1
assert answer_third_max([3, 2, 1]) == 1
assert answer_third_max([1, 2]) == 2
assert answer_third_max([2, 2, 3, 1]) == 1
assert answer_third_max([-5, -1, -3]) == -5
assert answer_dominant_by_sum([1, 2, 6]) == 2
assert answer_dominant_by_sum([1, 2, 2, 4]) == -1
assert answer_dominant_by_sum([5, 0, 0]) == 0
assert answer_max_product_three([1, 2, 3]) == 6
assert answer_max_product_three([1, 2, 3, 4]) == 24
assert answer_max_product_three([-10, -10, 1, 3, 2]) == 300
print("Tất cả đáp án PASS")

nums = [4, 9, 2, 4, 1]
i=0 x=  4 | 4 > first=-1 -> max mới, -1 xuống hạng 2 | first=  4 (i=0) second= -1
i=1 x=  9 | 9 > first=4 -> max mới, 4 xuống hạng 2   | first=  9 (i=1) second=  4
i=2 x=  2 | bỏ qua                                   | first=  9 (i=1) second=  4
i=3 x=  4 | bỏ qua                                   | first=  9 (i=1) second=  4
i=4 x=  1 | bỏ qua                                   | first=  9 (i=1) second=  4
Kết luận: 9 >= 2*4 = 8? True -> trả 1
Tất cả đáp án PASS


Ghi chú bài 4: dùng `None` làm "chưa có" và kiểm `is None` (không dùng `== None`). Câu `if x in (a, b2, c): continue` bỏ qua giá trị đã đứng trong top-3 → chỉ đếm giá trị **phân biệt**.

Ghi chú bài 6: tích lớn nhất là một trong hai: `top1·top2·top3` hoặc `top1·min1·min2` (hai số âm lớn về trị tuyệt đối).

### Template nhớ nhanh + Checklist submit

```python
class Solution:
    def dominantIndex(self, nums):
        first = second = -1
        first_i = -1
        for i, x in enumerate(nums):
            if x > first:                      # max mới
                second = first                 # max cũ xuống hạng 2 (TRƯỚC khi ghi đè)
                first, first_i = x, i
            elif x > second:                   # chỉ tranh hạng 2
                second = x
        return first_i if first >= 2 * second else -1   # >= : "at least twice"
```

**Câu thần chú:** *"so với mọi phần tử = so với hạng hai — giữ top-2 một lượt — trả chỉ số — `>=`."*

**Checklist trước khi submit:**
- [ ] Trả **chỉ số** của max, không phải giá trị?
- [ ] So với phần tử **khác** max (không so max với chính nó)?
- [ ] Dùng `>=` (vừa đúng gấp đôi vẫn hợp lệ)?
- [ ] Có nhánh `elif x > second` cho trường hợp max đứng trước hạng hai?
- [ ] Lưu `second = first` **trước** khi ghi đè `first`?
- [ ] Đã test `[1, 2]` → `1`, `[2, 3]` → `-1`, max ở đầu/cuối?